# 23. Pick a Reference Timestamp for Population Counting

<span style="font-family: 'Courier New', monospace;">

*AI-generated draft (Claude, Anthropic) — for review. All parameters and figures are derived from version-controlled scripts and data.*

The goal is to pick **one fixed second** where the camera is zoomed in on the Mushroom vent with worms clearly visible. That timestamp will be used to extract one consistent frame per video across the full dataset.

**Two-step process:**
1. **Wide scan (Cell 1)** — one frame every 60 seconds across the full video. Find the minute where the camera is zoomed onto the vent chimney.
2. **Fine scan (Cell 2)** — one frame every 5 seconds within that minute. Pick the exact second that looks clearest with the most worms visible.

Then open `24_count_timeseries.ipynb` and plug in that timestamp.

</span>

In [ ]:
import subprocess
import tempfile
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.image as mpimg

# ── Configuration ──────────────────────────────────────────────────
REFERENCE_VIDEO = Path(
    "/home/jovyan/ooi/san_data/RS03ASHS-PN03B-06-CAMHDA301/"
    "2024/10/01/CAMHDA301-20241001T031500.mp4"
)
SCAN_INTERVAL_SEC = 60   # one frame every 60 seconds
VIDEO_DURATION_SEC = 1500  # ~25 minutes; increase if video is longer
# ──────────────────────────────────────────────────────────────────

timestamps = list(range(0, VIDEO_DURATION_SEC, SCAN_INTERVAL_SEC))

print(f"Wide scan: {len(timestamps)} frames, one every {SCAN_INTERVAL_SEC}s")
print(f"Video: {REFERENCE_VIDEO.name}\n")

frames = {}
with tempfile.TemporaryDirectory() as tmpdir:
    for t in timestamps:
        out = Path(tmpdir) / f"frame_{t:05d}.png"
        subprocess.run(
            ["ffmpeg", "-y", "-ss", str(t), "-i", str(REFERENCE_VIDEO),
             "-frames:v", "1", "-q:v", "2", str(out)],
            capture_output=True, timeout=30,
        )
        if out.exists():
            frames[t] = mpimg.imread(str(out))

print(f"Extracted {len(frames)} frames. Look for the minute where the camera")
print("is zoomed in on the vent chimney — that's your target window.\n")

ncols = 4
nrows = (len(frames) + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(20, nrows * 3.2))
axes = axes.flatten()

for i, (t, img) in enumerate(sorted(frames.items())):
    axes[i].imshow(img)
    m, s = divmod(t, 60)
    axes[i].set_title(f"{m}m {s:02d}s  (t={t}s)", fontsize=11, fontweight="bold")
    axes[i].axis("off")

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle(
    f"Wide scan — {REFERENCE_VIDEO.name}\n"
    "Find the minute where the camera is zoomed onto the vent",
    fontsize=13, y=1.01,
)
plt.tight_layout()
plt.show()

print("Once you spot the right minute, set ZOOM_WINDOW_START in Cell 2 below.")

In [ ]:
# ── Fine scan — set ZOOM_WINDOW_START to the minute you identified above ──
ZOOM_WINDOW_START = 300   # ← UPDATE THIS to the start of the zoomed-in minute
ZOOM_WINDOW_END   = ZOOM_WINDOW_START + 60
FINE_INTERVAL_SEC = 5     # one frame every 5 seconds

timestamps_fine = list(range(ZOOM_WINDOW_START, ZOOM_WINDOW_END, FINE_INTERVAL_SEC))

print(f"Fine scan: {len(timestamps_fine)} frames, one every {FINE_INTERVAL_SEC}s")
print(f"Window: {ZOOM_WINDOW_START}s – {ZOOM_WINDOW_END}s\n")

frames_fine = {}
with tempfile.TemporaryDirectory() as tmpdir:
    for t in timestamps_fine:
        out = Path(tmpdir) / f"frame_{t:05d}.png"
        subprocess.run(
            ["ffmpeg", "-y", "-ss", str(t), "-i", str(REFERENCE_VIDEO),
             "-frames:v", "1", "-q:v", "2", str(out)],
            capture_output=True, timeout=30,
        )
        if out.exists():
            frames_fine[t] = mpimg.imread(str(out))

ncols = 4
nrows = (len(frames_fine) + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(20, nrows * 3.5))
axes = axes.flatten()

for i, (t, img) in enumerate(sorted(frames_fine.items())):
    axes[i].imshow(img)
    m, s = divmod(t, 60)
    axes[i].set_title(f"{m}m {s:02d}s  (t={t}s)", fontsize=12, fontweight="bold")
    axes[i].axis("off")

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle(
    f"Fine scan — {ZOOM_WINDOW_START}s to {ZOOM_WINDOW_END}s\n"
    "Pick the frame with the clearest zoom and most worms visible",
    fontsize=13, y=1.01,
)
plt.tight_layout()
plt.show()

print("\nNote your chosen timestamp — plug it into 24_count_timeseries.ipynb as REFERENCE_TIMESTAMP.")